In [1]:
!pip install -U transformers
!pip install accelerate -q


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [2]:
%%capture
!pip install --no-deps --upgrade timm # Only for Gemma 3N

In [3]:
!pip install opencv-python


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [4]:
!pip uninstall torchcodec -y

In [ ]:
from huggingface_hub import login
import os

# Paste your token here (get it from: https://huggingface.co/settings/tokens)
HF_TOKEN = "" 

login(token=HF_TOKEN)

In [6]:
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch

# Load the 2B finetuned model using transformers directly
processor = AutoProcessor.from_pretrained("blind-assist/google-gemma-3n-2b-e3")
model = AutoModelForImageTextToText.from_pretrained(
    "blind-assist/google-gemma-3n-2b-e3",
    torch_dtype=torch.bfloat16,
    device_map="cuda"
)

print(f"Model loaded on: {model.device}")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/1556 [00:00<?, ?it/s]

Model loaded on: cuda:0


In [ ]:
import cv2
import numpy as np
from PIL import Image

def downsample_video(video_path, num_frames=2):
    """Extracts evenly spaced frames for VLM context."""
    
    vidcap = cv2.VideoCapture(video_path)
    total_frames = int(vidcap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0: return []
    
    indices = np.linspace(0, total_frames - 1, num_frames, dtype=int)
    frames = []
    for i in indices:
        vidcap.set(cv2.CAP_PROP_POS_FRAMES, i)
        success, image = vidcap.read()
        if success:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            # Example: resize to 512x512
            frames.append(Image.fromarray(image))
    vidcap.release()
    return frames

In [ ]:
from datasets import load_dataset, Video, Dataset
from huggingface_hub import hf_hub_download
from itertools import islice
import os
import time
import csv

# Load test dataset from HuggingFace (streaming)
dataset_stream = load_dataset(
    "blind-assist/walk",
    split="test",
    streaming=True
)

dataset_stream = dataset_stream.cast_column("video", Video(decode=False))

# Download first 50 test videos locally
SAVE_DIR = "test_videos"
os.makedirs(SAVE_DIR, exist_ok=True)

N = 50
local_data = []

for item in islice(dataset_stream, N):
    video_hf_path = item["video"]["path"]
    filename = video_hf_path.split("/")[-1]
    local_path = os.path.join(SAVE_DIR, "test", filename)

    if not os.path.exists(local_path):
        hf_hub_download(
            repo_id="blind-assist/walk",
            filename=f"test/{filename}",
            repo_type="dataset",
            local_dir=SAVE_DIR,
            local_dir_use_symlinks=False
        )

    item["video"]["path"] = local_path
    local_data.append(item)

test_dataset = Dataset.from_list(local_data)
print(f"Loaded {len(test_dataset)} test samples")


In [ ]:
# Output CSV file
output_csv = "video_guidance_results_2b.csv"

# Instruction for the model
instruction = ("Given the visual input from the user's forward perspective, "
               "generate exactly one short sentence to guide a visually impaired user "
               "by identifying critical obstacles or landmarks, describing their locations "
               "using clock directions relative to the user (12 o\'clock is straight ahead), "
               "including relevant details such as size, material, or distance, and giving "
               "one clear action, while prioritizing immediate safety and avoiding any extra explanation.")

# Prepare results
results = []

print(f"Processing {len(test_dataset)} test videos")

# Process each video
for idx in range(len(test_dataset)):
    sample = test_dataset[idx]
    video_path = sample["video"]["path"]
    ground_truth = sample["alter"]
    video_filename = video_path.split("/")[-1]

    print(f"\n[{idx+1}/{len(test_dataset)}] Processing: {video_filename}")

    try:
        # Extract frames from video
        frames = downsample_video(video_path)

        if not frames:
            raise ValueError("No frames extracted from video")

        # Build the message with images and text
        content = []
        for img in frames:
            content.append({"type": "image", "image": img})
        content.append({"type": "text", "text": instruction})

        messages = [{"role": "user", "content": content}]

        # Process inputs using apply_chat_template
        inputs = processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        ).to(model.device)

        inputs["pixel_values"] = inputs["pixel_values"].to(model.dtype)

        # Generate output
        print(f"Generating guidance for {video_filename}...")

        start_time = time.time()

        output = model.generate(
            **inputs,
            max_new_tokens=128,
            # use_cache=True,
            # temperature=1.0,
            # top_p=0.95,
            # top_k=64
        )

        end_time = time.time()
        inference_time = round(end_time - start_time, 3)

        # Decode only the generated tokens (excluding the input)
        generated_text = processor.decode(output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

        # Clean up the output
        guidance = " ".join(generated_text.strip().split())

        print(f"Prediction:   {guidance}")
        print(f"Ground Truth: {ground_truth}")

        # Store result
        results.append({
            "video_id": idx + 1,
            "video_filename": video_filename,
            "prediction": guidance,
            "ground_truth": ground_truth,
            "inference_time_s": inference_time
        })

    except Exception as e:
        import traceback
        print(f"Error processing {video_filename}: {str(e)}")
        traceback.print_exc()
        results.append({
            "video_id": idx + 1,
            "video_filename": video_filename,
            "prediction": f"ERROR: {str(e)}",
            "ground_truth": ground_truth,
            "inference_time_s": 0.0
        })

# Save results to CSV
print(f"\nSaving results to {output_csv}...")
with open(output_csv, 'w', newline='', encoding='utf-8') as csvfile:
    fieldnames = ['video_id', 'video_filename', 'prediction', 'ground_truth', 'inference_time_s']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(results)

print(f"\nProcessing complete! Results saved to {output_csv}")
print(f"Successfully processed {len([r for r in results if not r['prediction'].startswith('ERROR')])} out of {len(test_dataset)} videos")


In [9]:
# Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA A40. Max memory = 44.422 GB.
10.979 GB of memory reserved.


In [ ]:
# Display results preview
import pandas as pd
df = pd.read_csv(output_csv)
print(df.head(10))
print(f"\nAverage inference time: {df['inference_time_s'].mean():.3f}s")
